## Check that the Window Dataset class works

In [3]:
import os
import pandas as pd
from tqdm import tqdm
if os.getcwd().endswith("notebooks"):
    os.chdir("../")

from src.embeddings.dataset import WindowDataset
from src.embeddings.train import train_ae

In [4]:
processed_path = "data/processed"
paths = [os.path.join(processed_path, f) for f in os.listdir(processed_path) if f.endswith("10fps_processed.pkl")]

df_dict = {}
cols_to_keep = ['frame', 'hand_label', 'cx_smooth', 'cy_smooth']

with tqdm(total=len(paths), desc="Loading processed data") as pbar:
    for path in paths:
        vid = os.path.basename(path).replace("_10fps_processed.pkl", "")
        vid = vid.replace("hand_tracking_", "")
        df_dict[vid] = pd.read_pickle(path)[cols_to_keep]
        pbar.update(1)

Loading processed data:   1%|          | 1/86 [00:00<00:27,  3.14it/s]0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Loading processed data: 100%|██████████| 86/86 [00:11<00:00,  7.55it/s]


In [5]:
dataset = WindowDataset(df_dict, hand="Right", feature_mode="pos_vel", window_size=20, step_size=5, orig_fps=30.0, max_gap_seconds=0.11, normalize=True)

Processing videos:   0%|          | 0/86 [00:00<?, ?it/s]

Processing videos: 100%|██████████| 86/86 [00:02<00:00, 29.38it/s]


# Training of the Autoencoder

In [6]:
ae = train_ae(dataset, feature_dim=4, latent_dim=16,
             batch_size=32, epochs=80, lr=1e-3, device="cpu")

Epoch 1/80 — Loss: 0.003730
Epoch 2/80 — Loss: 0.002503
Epoch 3/80 — Loss: 0.002484
Epoch 4/80 — Loss: 0.002472
Epoch 5/80 — Loss: 0.002457
Epoch 6/80 — Loss: 0.002389
Epoch 7/80 — Loss: 0.002332
Epoch 8/80 — Loss: 0.002320
Epoch 9/80 — Loss: 0.002315
Epoch 10/80 — Loss: 0.002312
Epoch 11/80 — Loss: 0.002310
Epoch 12/80 — Loss: 0.002307
Epoch 13/80 — Loss: 0.002306
Epoch 14/80 — Loss: 0.002305
Epoch 15/80 — Loss: 0.002303
Epoch 16/80 — Loss: 0.002303
Epoch 17/80 — Loss: 0.002301
Epoch 18/80 — Loss: 0.002300
Epoch 19/80 — Loss: 0.002298
Epoch 20/80 — Loss: 0.002298
Epoch 21/80 — Loss: 0.002296
Epoch 22/80 — Loss: 0.002296
Epoch 23/80 — Loss: 0.002296
Epoch 24/80 — Loss: 0.002294
Epoch 25/80 — Loss: 0.002294
Epoch 26/80 — Loss: 0.002292
Epoch 27/80 — Loss: 0.002294
Epoch 28/80 — Loss: 0.002293
Epoch 29/80 — Loss: 0.002293
Epoch 30/80 — Loss: 0.002292
Epoch 31/80 — Loss: 0.002292
Epoch 32/80 — Loss: 0.002292
Epoch 33/80 — Loss: 0.002292
Epoch 34/80 — Loss: 0.002291
Epoch 35/80 — Loss: 0.0

In [7]:
# save the trained autoencoder model
import torch
model_path = "results/models/motion_autoencoder1.0.pth"
torch.save(ae.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to results/models/motion_autoencoder1.0.pth
